In [ ]:
!pip install -q torch datasets tokenizers 2>/dev/null || echo "deps assumed preinstalled"
print('deps ready')

In [ ]:
import os
if not os.path.exists('/kaggle/working/pg/config/sft.json'):
    !git clone https://github.com/BayanDrp/pisto-gpt-64m.git /kaggle/working/pg
    print('cloned')
else:
    print('already present')
print('repo ready:', os.path.exists('/kaggle/working/pg/config/sft.json'))

In [ ]:
import os, sys
sys.path.insert(0, '/kaggle/working/pg')
from llm.tokenizer import ByteTokenizer
t = ByteTokenizer()
print('tokenizer vocab:', t.vocab_size)
assert t.vocab_size == 8192
print('tokenizer OK - using repo tokenizer (matches pretrained checkpoint)')

In [ ]:
import json, os
cfg = json.load(open('/kaggle/working/pg/config/sft.json'))
print('max_hours:', cfg['training']['max_hours'], '| max_steps:', cfg['training']['max_steps'], '| lr:', cfg['training']['lr'])
print('data sources:', [s['name'] for s in cfg['dataset']['sources']])

In [ ]:
import glob, os, shutil
W = '/kaggle/working/pg/weights'
os.makedirs(W, exist_ok=True)
for stale in glob.glob(f'{W}/instruct_*.pt') + [f'{W}/instruct_log.jsonl']:
    if os.path.exists(stale):
        os.remove(stale); print('removed stale', os.path.basename(stale))
if os.path.exists(f'{W}/pretrain_best.pt'):
    print('Using existing weights/pretrain_best.pt (already present — not overwriting)')
else:
    cands = glob.glob('/kaggle/input/**/pretrain_best.pt', recursive=True)
    if cands:
        shutil.copy(cands[0], f'{W}/pretrain_best.pt'); print('Copied pretrained weights from', cands[0])
    else:
        print('WARNING: pretrain_best.pt not found in /kaggle/input')

In [ ]:
import subprocess, sys
print('Building SFT dataset (downloading Arabic instruction corpora)...')
r = subprocess.run([sys.executable, '-u', 'scripts/build_sft_data.py'], cwd='/kaggle/working/pg')
print('build exit code:', r.returncode)

In [ ]:
import subprocess, sys
print('Starting SFT (auto-stops on overfit, max %d h)...' % 6)
r = subprocess.run([sys.executable, '-u', 'training/sft.py'], cwd='/kaggle/working/pg')
print('SFT exit code:', r.returncode)

In [ ]:
import glob, os
best = glob.glob('/kaggle/working/pg/weights/instruct_best.pt')
print('instruct_best.pt saved:', bool(best))
if best:
    print('Path:', best[0], '| download from notebook Output tab (pg/weights/instruct_best.pt)')
    # optional quick smoke test
    import sys; sys.path.insert(0, '/kaggle/working/pg')
    from llm.generate import chat
    for q in ['ما عاصمة فرنسا؟', 'كم حاصل 7 زائد 6؟']:
        print('Q:', q, '->', chat(q))